##Read both Silver tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
products = spark.table(
    "e2e_project.silver.crm_prd_info"
)

categories = spark.table(
    "e2e_project.silver.erp_product_category"
)

In [0]:
display(products.limit(20))
display(categories.limit(20))

In [0]:
products.printSchema()
categories.printSchema()

## Important: product history

In [0]:
current_products = products.filter(
    F.col("prd_end_dt").isNull()
)

display(current_products.limit(20))

This is an important modeling decision.

We're effectively saying:

dim_product represents the current known state of every product.

## Check how many current products exist

In [0]:
print("All Silver product rows:", products.count())
print("Current products:", current_products.count())

## Check category matching before joining

In [0]:
unmatched_categories = (
    current_products
    .join(
        categories,
        current_products["cat_id"] ==
        categories["category_id"],
        "left_anti"
    )
)

print(
    "Products without ERP category:",
    unmatched_categories.count()
)

display(unmatched_categories.limit(20))


left_anti
=
show rows from the left
without a matching row on the right

## Join CRM products with ERP categories

In [0]:
product_joined = (
    current_products.alias("prd")
    .join(
        categories.alias("cat"),
        F.col("prd.cat_id") ==
        F.col("cat.category_id"),
        "left"
    )
)

Every CRM product
+
ERP category information when available

## Build clean Gold columns

In [0]:
dim_product = product_joined.select(

    F.col("prd.prd_id").alias("product_id"),

    F.col("prd.prd_key").alias("product_number"),

    F.col("prd.prd_nm").alias("product_name"),

    F.col("prd.cat_id").alias("category_id"),

    F.coalesce(
        F.col("cat.category"),
        F.lit("n/a")
    ).alias("category"),

    F.coalesce(
        F.col("cat.subcategory"),
        F.lit("n/a")
    ).alias("subcategory"),

    F.coalesce(
        F.col("cat.maintenance"),
        F.lit("n/a")
    ).alias("maintenance"),

    F.col("prd.prd_cost").alias("cost"),

    F.col("prd.prd_line").alias("product_line"),

    F.col("prd.prd_start_dt").alias("start_date")
)

## Add a surrogate product key

In [0]:
product_window = Window.orderBy(
    "start_date",
    "product_number"
)

dim_product = dim_product.withColumn(
    "product_key",
    F.row_number().over(product_window)
)

In [0]:
dim_product = dim_product.select(
    "product_key",
    "product_id",
    "product_number",
    "product_name",
    "category_id",
    "category",
    "subcategory",
    "maintenance",
    "cost",
    "product_line",
    "start_date"
)

## Inspect your dimension

In [0]:
display(dim_product.limit(50))

In [0]:
dim_product.printSchema()

## Validate uniqueness

In [0]:
display(
    dim_product
    .groupBy("product_key")
    .count()
    .filter(F.col("count") > 1)
)

In [0]:
display(
    dim_product
    .groupBy("product_number")
    .count()
    .filter(F.col("count") > 1)
)

## Validate the join didn't multiply rows

In [0]:
print(
    "Current Silver products:",
    current_products.count()
)

print(
    "Gold products:",
    dim_product.count()
)

That means your join probably created duplicate matches.

That is called row multiplication, and it's one of the most common data-engineering mistakes.

This is why we always validate row counts after joins.

## Check NULLs

In [0]:
display(
    dim_product.select(
        [
            F.sum(
                F.col(c).isNull().cast("int")
            ).alias(c)
            for c in dim_product.columns
        ]
    )
)

## Validate cost

In [0]:
display(
    dim_product.filter(
        F.col("cost").isNull() |
        (F.col("cost") < 0)
    )
)

## Save dim_product

In [0]:
(
    dim_product.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "e2e_project.gold.dim_product"
    )
)

In [0]:
%sql

SELECT *
FROM e2e_project.gold.dim_product
LIMIT 20;

So Gold is doing integration, not merely another cleaning pass.

![image_1788829459695.png](./image_1788829459695.png "image_1788829459695.png")